# 00 — Dicionário de dados

**CRISP-DM fase 2.2 — descrição dos dados** — Segmentação do mercado de veículos usados de Fortaleza

---

Este notebook monta o **dicionário de dados da base bruta**: para cada uma das
23 colunas de `data/raw/anuncios_detalhados.csv`, registra o nome da coluna,
uma descrição de negócio, a faixa (ou os extremos) dos valores observados e o
tipo de variável.

Diferente do projeto de referência (Ames Housing), aqui **não há download de
uma base pública**: a camada bruta é o resultado direto do scraping da OLX
(ver `Software/coleta/`), já materializada em `data/raw/` e tratada como
imutável a partir deste ponto.

In [ ]:
# importações
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

from src import config

## Carregando a base de dados

A fonte é `data/raw/anuncios_detalhados.csv`: 2.565 anúncios únicos de
Fortaleza-CE, coletados por scraping da OLX e detalhados a 100% (ver
`Software/coleta/HISTORICO.md` para o histórico completo da coleta).

In [ ]:
# Carregando base de dados
df = pd.read_csv(config.ARQUIVO_BRUTO)
origem = config.ARQUIVO_BRUTO

print(f"{origem.relative_to(RAIZ)} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

## Varredura das colunas

Percorremos coluna a coluna classificando o tipo e coletando os valores de
referência:

| dtype | tipo atribuído | valores registrados |
|---|---|---|
| inteiro | Discreta | mínimo e máximo |
| decimal | Contínua | mínimo e máximo |
| texto | Nominal | quantidade de categorias e exemplos |

Diferente da base Ames Housing, esta base **não tem colunas com escala
ordinal declarada** (tipo `Ex > Gd > TA > Fa > Po`) — todas as colunas de
texto são tratadas como Nominais nesta etapa.

In [ ]:
# lista de dicionários para armazenar informações sobre as colunas
col_info = []


# Converte escalar numpy em escalar Python. Sem isso o CSV sai com
# `[np.int64(12789), np.int64(755000)]` em vez de `[12789, 755000]` — é só o
# `repr` do numpy 2, mas o dicionário é lido por gente, não por máquina.
def escalar(valor):
    return valor.item() if hasattr(valor, "item") else valor


# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]
    n_nulos = int(serie.isna().sum())

    if pd.api.types.is_integer_dtype(serie):
        tipo = "Discreta"
        valores = [escalar(serie.min()), escalar(serie.max())]
    elif pd.api.types.is_float_dtype(serie):
        tipo = "Contínua"
        valores = [escalar(serie.min()), escalar(serie.max())]
    else:
        tipo = "Nominal"
        categorias = sorted(serie.dropna().unique().tolist())
        valores = categorias if len(categorias) <= 10 else categorias[:10] + ["…"]

    col_info.append(
        {
            "Variável": col,
            "Tipo": tipo,
            "Valores/Faixa": valores,
            "Nulos": n_nulos,
            "% nulos": round(100 * n_nulos / len(df), 1),
        }
    )

In [ ]:
# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

print(f"{len(col_info_df)} colunas descritas")
col_info_df["Tipo"].value_counts()

## Descrições de negócio

Cada coluna recebe uma descrição em linguagem de negócio, casada por nome —
não por posição — para não quebrar em silêncio se a ordem das colunas mudar.

In [ ]:
# Descrições de negócio, por nome da coluna
DESCRICOES = {
    "titulo": "Título do anúncio, como publicado pelo vendedor na OLX",
    "preco": "Preço anunciado, em reais (R$)",
    "localizacao": "Localização declarada no card de listagem (cidade, bairro)",
    "km": "Quilometragem informada no anúncio",
    "cor": "Cor do veículo",
    "motor": "Motorização declarada no card de listagem (ex.: '1.6')",
    "tipo": "Tipo/carroceria conforme o card de listagem",
    "ano": "Ano do veículo, extraído do título do anúncio",
    "link": "URL do anúncio na OLX (chave única do anúncio)",
    "valor_fipe_olx": "Valor de referência FIPE mostrado pela própria OLX no anúncio, quando disponível",
    "marca": "Marca do veículo, extraída do dataLayer da página do anúncio",
    "modelo": "Modelo do veículo (dataLayer)",
    "versao": "Versão/acabamento do veículo (dataLayer)",
    "cambio": "Tipo de câmbio (Manual, Automático, Semi-Automático, Automatizado)",
    "combustivel": "Tipo de combustível (Gasolina, Flex, Álcool, Diesel, Híbrido, Elétrico)",
    "carroceria": "Carroceria do veículo, conforme o dataLayer (Sedã, Hatch, SUV, Pick-up…)",
    "portas": "Número de portas do veículo",
    "bairro": "Bairro onde o anunciante está localizado",
    "municipio": "Município onde o anunciante está localizado",
    "vendedor_tipo": "Tipo de vendedor: 'particular' ou 'profissional' (loja/revenda)",
    "aceita_troca": "Indica se o vendedor aceita troca por outro veículo (Sim/Não)",
    "unico_dono": "Indica se o veículo é de único dono (Sim/Não)",
    "data_coleta": "Data em que o anúncio foi coletado (scraping)",
}

len(DESCRICOES)

In [ ]:
# Adicionando descrição para cada uma das variáveis, casada por nome de coluna
col_info_df["Descrição"] = col_info_df["Variável"].map(DESCRICOES)

faltando = col_info_df[col_info_df["Descrição"].isna()]
assert faltando.empty, f"Colunas sem descrição: {faltando['Variável'].tolist()}"

col_info_df

## Salvando o dicionário de dados

O arquivo vai para `data/processed/dicionario.csv`, separado por `;` e sem
índice — o mesmo formato consumido pelos demais notebooks.

> As descrições da **camada curada** (colunas derivadas/agregadas na etapa de
> preparação) vivem em `src/data/dicionario.py`. Este CSV descreve a base
> **bruta**: é o retrato de entrada, anterior a qualquer transformação.

In [ ]:
# Salvando dicionário de dados
config.garantir_diretorios()
destino = config.DATA_PROCESSED / "dicionario.csv"
col_info_df.to_csv(destino, index=False, sep=";")

print(f"Dicionário gravado em {destino.relative_to(RAIZ)} ({destino.stat().st_size:,} bytes)")
pd.read_csv(destino, sep=";").head(10)